In [24]:
import numpy as np
import astropy.units as u
from astropy.io import fits
import time
from datetime import datetime
today = int(datetime.today().strftime('%Y%m%d'))
from skimage.registration import phase_cross_correlation
import copy
from importlib import reload
from IPython.display import clear_output, display
from pathlib import Path
import os
cwd = Path(os.getcwd())

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, CenteredNorm
from matplotlib.patches import Circle

import magpyx
from magpyx.utils import ImageStream
import purepyindi
from purepyindi import INDIClient
import purepyindi2
from purepyindi2 import IndiClient

client0 = INDIClient('localhost', 7624)
client0.start()
client = IndiClient()
client.connect()
client.get_properties()

import lina
from lina.math_module import xp, xcipy, ensure_np_array
from lina import utils, rt_utils, coro_utils

import fsm_utils
import cam_utils

v_bias = np.array([[50,50,50]]).T
zero = np.array([[0,0,0]]).T

wavelength = 633e-9
fl = 500e-3
fsm_pupil_diam = 6.7e-3
rad_per_lamD = wavelength/fsm_pupil_diam
print(rad_per_lamD)
as_per_lamD = (wavelength/fsm_pupil_diam*u.radian).to_value(u.arcsec)
print(as_per_lamD)

pxscl_lamD = 3.45e-6 / (fl * wavelength/fsm_pupil_diam)
print(pxscl_lamD)

9.44776119402985e-05
19.48740632155403
0.07303317535545024


INFO:purepyindi2.transports:Opening localhost:7624...
INFO:purepyindi2.transports:Connected to localhost:7624


In [27]:
camfsm_telem_path = Path(f'/opt/MagAOX/rawimages/camfsm/camfsm/2026_09_25')
camfsm_data_path = Path(f'camfsm-data-{today}')
utils.make_dir(camfsm_data_path)

dm01_telem_path = Path(f'/opt/MagAOX/rawimages/dm00disp01/dm00disp01/2026_09_30')
dm01_data_path = Path(f'dm01-data-{today}')
utils.make_dir(dm01_data_path)



Directory 'camfsm-data-20260930' already exists.
Directory 'dm01-data-20260930' already exists.


In [ ]:
dmt_channel = 'dm00disp'
dm00_channel = 'dm00disp00'
dm01_channel = 'dm00disp01'

DMT_STREAM = ImageStream(dmt_channel)
DM00_STREAM = ImageStream(dm00_channel)
DM01_STREAM = ImageStream(dm01_channel)

zeros = np.array([[0, 0, 0]]).T
vals = np.array([[1, 0, 0]]).T
DM00_STREAM.write(zeros)

In [25]:
reload(rt_utils)


rt_utils.toggle_telem(1, DM01_STREAM.name, client0)

time.sleep(1)
for i in range(5):
    DM01_STREAM.write(zeros)
    time.sleep(0.5)

rt_utils.toggle_telem(0, DM01_STREAM.name, client0)

In [34]:
import shutil
def copy_files(src_dir, dst_dir):
    # os.makedirs(dst_dir, exist_ok=True)
    for file in os.listdir(src_dir):
        src_path = os.path.join(src_dir, file)
        if os.path.isfile(src_path):
            shutil.copy(src_path, dst_dir)

In [42]:
# fnames = utils.get_fnames(dm01_telem_path/'fits/*.fits')
fnames = utils.get_fnames(dm01_data_path/'*.fits')
fnames

['dm01-data-20260930/dm00disp01_20260930222412489065544.fits',
 'dm01-data-20260930/dm00disp01_20260930222412989233914.fits',
 'dm01-data-20260930/dm00disp01_20260930222413489874516.fits',
 'dm01-data-20260930/dm00disp01_20260930222413990033879.fits',
 'dm01-data-20260930/dm00disp01_20260930231837110624722.fits',
 'dm01-data-20260930/dm00disp01_20260930231837611251907.fits',
 'dm01-data-20260930/dm00disp01_20260930231838111857314.fits',
 'dm01-data-20260930/dm00disp01_20260930231838612480881.fits',
 'dm01-data-20260930/dm00disp01_20260930231839113120165.fits']

In [43]:
rt_utils.read_telem_times(fnames, absolute=False)

array([0.00000000e+00, 5.00168370e-01, 1.00080897e+00, 1.50096834e+00,
       3.26462156e+03, 3.26512219e+03, 3.26562279e+03, 3.26612342e+03,
       3.26662405e+03])

In [35]:
# utils.move_files(dm01_telem_path/'fits', dm01_data_path)
copy_files(dm01_telem_path/'fits', dm01_data_path)

In [9]:
rt_utils.unpack_data(
    camfsm_telem_path,
    camfsm_data_path,
)